# Project 3. Part 1 - CLD Aggregation Tool

Last update: 05/23/2024

Title: Streamlining the Aggregation of Causal Loop Diagrams: A Thematic Labeling and Adjacency Matrix Approach

Overview: 
Spaghetti Causal Loop Diagrams (CLDs) present a significant challenge for modelers in effectively communicating complex system dynamics to stakeholders. Traditionally, modelers manually synthesize multiple CLDs to create an aggregated diagram that provides a higher-level understanding of the system. This paper introduces a streamlined procedure for aggregating CLDs using general categorical labels. 

The methodology involves translating qualitative mental models into adjacency matrices that capture both links and polarity signs, manually coding and substituting variable names with thematic labels, aggregating these matrices to identify significant feedback loops, and validating the approach by comparing it with the manual synthesis performed by an expert system dynamicist. 

Our results demonstrate that this tool is highly useful for group model-building experts, particularly when dealing with a large number of mental maps. The proposed method enhances the clarity and efficiency of synthesizing complex CLDs, thereby improving the overall communication of system dynamics.







In [100]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# READ I/O

In [101]:

from spellchecker import SpellChecker
import numpy as np
import os 
import pickle
import pandas as pd
import sys
import networkx as nx
import matplotlib.pyplot as plt
import gravis as gv


# Add the src directory to the sys.path
sys.path.append(os.path.abspath(os.path.join('..', 'src')))
print(sys.path)

from vensim_parser import CLDParser
# from cld_aggregator import CLDGraph 

import parse_cld
import cld_nx
import data_io


['C:\\Program Files\\Python310\\python310.zip', 'C:\\Program Files\\Python310\\DLLs', 'C:\\Program Files\\Python310\\lib', 'C:\\Program Files\\Python310', 'c:\\Users\\GL665\\Documents\\Phrame-MentalMaps\\venv', '', 'c:\\Users\\GL665\\Documents\\Phrame-MentalMaps\\venv\\lib\\site-packages', 'c:\\Users\\GL665\\Documents\\Phrame-MentalMaps\\venv\\lib\\site-packages\\win32', 'c:\\Users\\GL665\\Documents\\Phrame-MentalMaps\\venv\\lib\\site-packages\\win32\\lib', 'c:\\Users\\GL665\\Documents\\Phrame-MentalMaps\\venv\\lib\\site-packages\\Pythonwin', 'c:\\Users\\GL665\\Documents\\Phrame-MentalMaps', 'c:\\Users\\GL665\\Documents\\Phrame-MentalMaps', 'c:\\Users\\GL665\\Documents\\Phrame-MentalMaps', 'c:\\Users\\GL665\\Documents\\Phrame-MentalMaps', 'c:\\Users\\GL665\\Documents\\Phrame-MentalMaps', 'c:\\Users\\GL665\\Documents\\Phrame-MentalMaps', 'c:\\Users\\GL665\\Documents\\Phrame-MentalMaps', 'c:\\Users\\GL665\\Documents\\Phrame-MentalMaps', 'c:\\Users\\GL665\\Documents\\Phrame-MentalMaps', '

# Read Data

## read raw data

In [102]:
# Set the new directory
new_dir = "/Users/gl665/PycharmProjects/Phrame_MentalMaps"
# Change the current working directory
os.chdir(new_dir)

print(os.getcwd())

FileNotFoundError: [WinError 3] The system cannot find the path specified: '/Users/gl665/PycharmProjects/Phrame_MentalMaps'

In [ ]:
# Read all the json files in 
my_data_RC = {}
folder_path = 'results/SYNCH/JSON_file/RC/'

my_data_RC = data_io.read_json(folder_path, my_data_RC)


load: FM 51220Clean.json
load: Passages Clean.json
load: full circle GBM 5 24 21clean.json
load: C 7 6 20 clean model.json
load: Wavi combined clean.json
load: drug justice model clean.json
load: DSS23.json
load: E clean model 572020.json
load: nat model clean2.json
load: FC Prov Clean.json
load: full circle clean.json
load: reentry clean model.json
load: DV justice model clean.json


## format variables and spell checking

enculteration???  -> Acculturation


# write out the formated and cleaned data to pickle

In [ ]:
# import pickle

# # Specify the file path where you want to save the pickled data
file_path = "results/SYNCH/JSON_file/RC_cld_data_cleaned_preprocessed.json"

# Pickle dump the cld_data_new dictionary
with open(file_path, "wb") as file:
    pickle.dump(my_data_RC, file)


## Read Preprocessed data

In [ ]:
# read RC data after cleaning and merging the semantic similar variable names 
# my_data_RC = {}
# # Specify the file path
# file_path = 'results/SYNCH/JSON_file/RC_cld_data_merged.json'

# # Read the JSON file
# with open(file_path, 'rb') as file:
#     my_data_RC = pickle.load(file)

# # Print the data
# print(my_data_RC)

## Read file_rename data

In [ ]:
file_rename = pd.read_excel('results/SYNCH/file_rename/RC_file_rename.xlsx', index_col=0)
file_rename['file_no.'] = [i+1 for i in range(len(file_rename.index))]

dict_file_rename = dict(zip(file_rename['Filename'], file_rename['Model Name']))
dict_file_rename

{'FM 51220Clean.json': 'FM',
 'Passages Clean.json': 'PA',
 'full circle GBM 5 24 21clean.json': 'FullGMB',
 'C 7 6 20 clean model.json': 'C76',
 'tracy clean model.json': 'Tracy',
 'Wavi combined clean.json': 'Wavi',
 'drug justice model clean.json': 'JUSTICE',
 '7 13 20 clean model.json': 'C713',
 'DSS23.json': 'DSS',
 'E clean model 572020.json': 'E',
 'nat model clean2.json': 'NAT',
 'j model 7 13 20 clean model3.json': 'J713',
 'FC Prov Clean.json': 'FC',
 'full circle clean.json': 'Full',
 'reentry clean model.json': 'Reentry',
 '1 6 21 Legal clean.json': 'Legal',
 'DV justice model clean.json': 'DV',
 'SOC clean.json': 'SOC',
 'RC 4 28 2021 Clean.json': 'RC'}

## Load all the graphs in a dictionary, {cld_name: networkX graph}

In [ ]:
# dict_cld = {name_file: nxgraph}
def get_cld_graph(dict_cld, cld_name = str):
    cld_list = dict_cld[cld_name]
    G = nx.from_dict_of_dicts(cld_list[1])
    G = nx.relabel_nodes(G, cld_list[0])

    return cld_list, G

In [ ]:
dict_cld = {}

for k, v in my_data_RC.items():
    cld_list, G_cld = get_cld_graph(my_data_RC, k)
    cld_name = dict_file_rename.get(k, "")

    dict_cld[cld_name] = G_cld

list_of_G = list(dict_cld.values())

In [ ]:
# get all variable names 
all_var_names = []

for G in list_of_G:
    all_var_names.extend(list(G.nodes))

all_var_names = list(set(all_var_names))    
all_var_names

['community-level trauma',
 'percieved efficacy of treatment',
 'outreach to young women on birth control planning',
 'stigma of being a \\bad parent\\',
 'stressful events',
 'mental health resources',
 'prison mental health',
 'child home functioning',
 'opportunities for arrest',
 'personal strength',
 'exposure to antisocial peers',
 'physical violence',
 'housing need',
 'available mental health in-jail resources',
 'overworking of police',
 'home healthy relationship education',
 'funding for programs',
 'community alc m rate',
 'coworker social support',
 'legal requirement for arrest',
 'child welfare report',
 'child in foster system',
 'desire for treatment',
 'unresolved ptsd/trauma',
 'healthy relationship skills',
 'dependence on family for resources',
 'reluctance to report to child welfare',
 'kids witness ipv',
 'resources for adjustment',
 'alc m',
 'ihs insurance qualifications for birth control',
 'terminating pregnancy',
 'using mental health in-jail resources',
 'j

## Read handcraft data

In [ ]:
folder_path = 'results/SYNCH/JSON_file/RC/'


input_file = 'data/SYNCH/handcraft_aggergate_model/AggregateModel_SubsystemColorsfullmodelcolors.mdl'

df =  data_io.read_cld_file(input_file)

cld = CLDParser(df)

cld_hand_craft = cld.get_cld_dict()


### format variables and spell checking

In [ ]:
custom_words = ['american indian and alaska native', 
                'indian', 
                'alaska',
                'christianity', 
                'fostercare', 
                'american',
                'child-rearing',
                'reoffending']


# clean variable format 
for key, value in cld_hand_craft[0].items():
    # print(value)
    value = cld_nx.clean_variable_name(value)
    # print(value)

    corrected_text = cld_nx.correct_spelling(value, custom_words)
    if value != corrected_text: 
        print(f'Original: {value}')
        print(f'Corrected: {corrected_text}')
    
    cld_hand_craft[0][key] = corrected_text    




Original: accessability of alcohol or drug use community prevention or intervention resources
Corrected: accessibility of alcohol or drug use community prevention or intervention resources
Original: accessability of reproductive and prenatal care
Corrected: accessibility of reproductive and prenatal care
Original: department of social service resource accessability
Corrected: department of social service resource accessibility
Original: intrapersonal skills for navigating department of social service
Corrected: interpersonal skills for navigating department of social service
Original: intimate partner violence survivor recieving or engaging in intimate partner violence victim resources
Corrected: intimate partner violence survivor receiving or engaging in intimate partner violence victim resources
Original: intimate partner violence survivor with fear of negative repercussions (losing child)
Corrected: intimate partner violence survivor with fear of negative repercussions losing child


In [ ]:
for key, value in cld_hand_craft[0].items():
    print(key, value)

1 acceptability of birth control
2 accessibility of alcohol or drug use community prevention or intervention resources
3 accessibility of reproductive and prenatal care
4 accessibility of alcohol or drug use treatment
5 accessibility of birth control
6 accessibility of resources for basic needs
7 accessibility of western resources for indigenous people
8 accountability
9 adherence to indian child welfare act
10 antisocial peer network
11 children experiencing abuse
12 children experiencing adverse childhood events
13 children exposed to violence
14 children placed in native foster care
15 children placed in non native fostercare
16 children placed with family
17 children removed from home
18 children with fetal alcohol spectrum disorders
19 colonial oppression and violence
20 community quality or safety
21 cultural competency or fit of care for american indian and alaska native intimate partner violence survivor
22 dependence on others to meet needs
23 desire to change
24 development o

# Compare List of Graphs variable names and Handcraft variable names 

Check if the two have a overlap. 


In [ ]:
# create a networkX graph from the handcraft model
g = nx.from_dict_of_dicts(cld_hand_craft[1])
g = nx.relabel_nodes( g, cld_hand_craft[0])


In [ ]:
# cld_nx.cld_nx_node_properties(g)
# cld_nx.cld_nx_edge_properties(g)
cld_nx.cld_nx_properties(g)


Graph data:
-------------
Type: <class 'networkx.classes.graph.Graph'>
Directed: False
Number of nodes: 149
Number of edges: 275


In [ ]:
# get all variable names
all_var_names_hand_craft = list(g.nodes)


print("all_var_names_hand_craft: ") 
for i in all_var_names_hand_craft:
    print(i)

all_var_names_hand_craft: 
healthfulness of ipv & violence behaviors and norms
family functioning
ipv stigma
healthfulness of child-rearing practices
people experiencing ipv
relationship functioning
children experiencing abuse
severity of incident
people arrested
people convicted of offense
people charged with offense
police involvement
perceived credibility of ipv report
ipv survivor with dependency on partner
trust in justice/police systems
effectiveness/quality of policing system
quality of social support systems
people with good mental health
people misusing or abusing drugs and alcohol
colonial oppression and violence
people who experience racism/violence by police and justice system
accessibility of birth control
ipv survivor using birth control or seeking abortion
acceptability of birth control
children placed in non-native fostercare
development of positive aian cultural identity
structural and institutional racism
likelihood of tribal involvement
people with protective coping 

# Compare List of Graphs variable names and Handcraft variable names 

Check if the two have a overlap. 

Result: there's only 9 variables overlapp between the handcraft CLD v.s. the list of individual graphs. that means the handcraft CLD is a merge of the individual CLDs however, the variable names does not match up exactly. 

Hence, there is no baseline, to compare the handcraft cld with the list of individual clds. 



In [ ]:
# check how much variables overlapped between the two list of all variable names 
overlap = [var for var in all_var_names if var in all_var_names_hand_craft]
print("overlapping variables between all the variables names used in the RC models and the handcraft model: ")

for i in overlap:
    print(i)



overlapping variables between all the variables names used in the RC models and the handcraft model: 
forced assimilation
historical trauma
severity of incident
police involvement
accountability
relationship functioning
family functioning
quality of prenatal healthcare
acceptability of birth control
numbers of overlapped variables:  9


In [ ]:
print("numbers of overlapped variables: ", len(overlap))

numbers of overlapped variables:  9


# Clean Themantic Data excel sheet

In [ ]:
# preprocess the RC_Updated_Themes.xslx and match it back with the RC files. 

df_rc_themes = pd.read_excel('data/SYNCH/aggregate_theme/Rapid_City_Updated_Themes_w_levels.xlsx', sheet_name= "clean")

# # get column 1 and 4
df_aggregate_names = df_rc_themes.iloc[:, [1, 3, 4, 5]]

df_aggregate_names.columns = ['theme', 'sub_theme', 'variable', 'level'] # TODO: use the subthem label when complete 


#  delete rows if all columns are nan 
df_aggregate_names = df_aggregate_names.dropna(subset=['theme', 'variable'], how='all')


df_aggregate_names['theme'] = df_aggregate_names['theme'].ffill()
# df_aggregate_names
df_aggregate_names['theme']= df_aggregate_names['theme'].str.lower()
df_aggregate_names['variable']= df_aggregate_names['variable'].str.lower()

# df_aggregate_names.head(20)
# df_aggregate_names.to_csv('Dataset/SYNCH/aggregate_theme/RC_theme_var_matching.csv', index=False)

In [ ]:
# create columns for different levels 
df_aggregate_names['personal'] = df_aggregate_names['level'].apply(lambda x: 'personal' in str(x).lower())
df_aggregate_names['institutional'] = df_aggregate_names['level'].apply(lambda x: 'institutional' in str(x).lower())
df_aggregate_names['relational'] = df_aggregate_names['level'].apply(lambda x: 'relational' in str(x).lower())
df_aggregate_names['cultural'] = df_aggregate_names['level'].apply(lambda x: 'cultural' in str(x).lower())
df_aggregate_names['community'] = df_aggregate_names['level'].apply(lambda x: 'community' in str(x).lower())
df_aggregate_names['societal'] = df_aggregate_names['level'].apply(lambda x: 'societal' in str(x).lower())


In [ ]:
df_aggregate_names.head(5)

,theme,sub_theme,variable,level,personal,institutional,relational,cultural,community,societal
1,healthy development,NaN,child psychological development,personal,True,False,False,False,False,False
2,healthy development,NaN,healthy child development,personal,True,False,False,False,False,False
3,healthy development,NaN,child behavioral issues,personal,True,False,False,False,False,False
4,healthy development,NaN,child physical violent perpetration,personal,True,False,False,False,False,False
5,healthy development,NaN,developmental delay in child,personal,True,False,False,False,False,False


In [ ]:
df_aggregate_names['level'].unique()

array(['personal', 'personal-institutional', 'personal-relational',
       'institutional', nan, 'relational',
       'personal-relational-institutional', 'relational-community',
       'relational-institutional', 'community',
       'personal-relational-community', 'personal-community',
       'institutional-community', 'personal-institutional-community',
       'personal-commuinty', 'personal-societal', 'community-cultural',
       'institutional-community-cultural',
       'personal-institutional-community-cultual', 'relational-cultural',
       'personal-community-cultural', 'relational-community-cultural',
       'institutional-cultural', 'community-institutional-cultural',
       'personal-institutional-cultural', 'personal-relational-cultural',
       'personal-cultural', 'personal-relational-community-cultural',
       'personal-instituional', 'instituitional-community'], dtype=object)

In [ ]:
# df_aggregate_names = pd.read_csv('data/SYNCH/aggregate_theme/RC_theme_var_matching.csv')
unique_theme_names = set(df_aggregate_names['theme'])
# df_aggregate_names.head()


In [ ]:
# df_aggregate_names = pd.read_csv('data/SYNCH/aggregate_theme/RC_theme_var_matching.csv')
# unique_theme_names = set(df_aggregate_names['theme'])
# df_aggregate_names.head()


# Add attributes of the levels to the nodes based on the themantic label

In [ ]:
# set node and cld attributes 
for k , v in dict_cld.items():
    nx.set_node_attributes(v, name = 'cld', values = k)
    nx.set_edge_attributes(v, name = 'cld', values = k)
    cld_nx.set_node_theme_attributes(v, unique_theme_names)


    

In [ ]:
def set_level_node_attribute(G, df_aggregate_names, level: str):
    mydict = {row['variable']: row[level] for _, row in df_aggregate_names.iterrows()}

    for node in G.nodes():
        if node in mydict:
            G.nodes[node][level] = 1 if mydict[node] else 0

    return G

for k, v in dict_cld.items():
    G1 = v

    G1 = set_level_node_attribute(G1, df_aggregate_names,  'personal')

    G1 = set_level_node_attribute(G1, df_aggregate_names,  'institutional')

    G1 = set_level_node_attribute(G1, df_aggregate_names,  'relational')

    G1 = set_level_node_attribute(G1, df_aggregate_names,  'cultural')

    G1 = set_level_node_attribute(G1, df_aggregate_names,  'community')




In [ ]:
# merge the node name based on the theme dict 

# for k , v in dict_cld.items():
#     v = nx.relabel_nodes(v, theme_dict)

In [ ]:
cld_nx.cld_nx_properties(dict_cld['FM'])
cld_nx.cld_nx_node_properties(dict_cld['FM'])
cld_nx.cld_nx_edge_properties(dict_cld['FM'])

Graph data:
-------------
Type: <class 'networkx.classes.graph.Graph'>
Directed: False
Number of nodes: 112
Number of edges: 273
Node data:
-------------

Type: <class 'numpy.str_'> <class 'dict'>
Var Name: alcohol misuse
Attributes: {'cld': 'FM', 'level': 'org', 'degree': 9, 'personal': 1, 'institutional': 0, 'relational': 0, 'cultural': 0, 'community': 0}
Degree: 9
Type: <class 'numpy.str_'> <class 'dict'>
Var Name: mental health
Attributes: {'cld': 'FM', 'level': 'theme', 'degree': 15, 'personal': 1, 'institutional': 0, 'relational': 0, 'cultural': 0, 'community': 0}
Degree: 15
Type: <class 'numpy.str_'> <class 'dict'>
Var Name: intimate partner violence
Attributes: {'cld': 'FM', 'level': 'org', 'degree': 16}
Degree: 16
Type: <class 'str'> <class 'dict'>
Var Name: abandonment
Attributes: {'cld': 'FM', 'level': 'org', 'degree': 7}
Degree: 7
Type: <class 'numpy.str_'> <class 'dict'>
Var Name: family emotional health
Attributes: {'cld': 'FM', 'level': 'org', 'degree': 8, 'personal': 0,

In [ ]:

# def clean_graph_for_gravis(graph):
#     # Convert all node and edge attributes to JSON serializable types
#     for node, attrs in graph.nodes(data=True):
#         for key, value in attrs.items():
            
#             if isinstance(value, np.integer):
#                 print(value)
#                 print(type(value))
#                 attrs[key] = int(value)
#             elif isinstance(value,  np.floating):
#                 attrs[key] = float(value)

#     for u, v, attrs in graph.edges(data=True):
#         for key, value in attrs.items():
#             if isinstance(value, np.integer):
#                 attrs[key] = int(value)
#                 print(attrs[key])
#             elif isinstance(value,  np.floating):
#                 attrs[key] = float(value)
#     return graph

In [ ]:
cld_nx.gravis_settings(dict_cld['FM'])


gv.d3(
    dict_cld['FM'], 
    graph_height=500)
#     edge_curvature=-0.5,
    
#     node_size_data_source="degree",
#     use_node_size_normalization=True,
#     node_size_normalization_min=15,
#     node_size_normalization_max=35,
#     show_node_label=True,
#     many_body_force_strength=-500
#  ) 



# Swap the node names to the theme names based on the themantic label 

# Merging by adding up adjency matrix (Self define version)

In [ ]:
# get combined graph                          
combined_graph = cld_nx.combine_graphs_with_node_attributes(list_of_G)

In [ ]:
clean_combined_graph = cld_nx.clean_combined_graph(combined_graph, 0, 0 )

cld_nx.cld_nx_properties(clean_combined_graph)
cld_nx.cld_nx_node_properties(clean_combined_graph)
cld_nx.cld_nx_edge_properties(clean_combined_graph)


Graph data:
-------------
Type: <class 'networkx.classes.digraph.DiGraph'>
Directed: True
Number of nodes: 754
Number of edges: 1474
Node data:
-------------

Type: <class 'numpy.str_'> <class 'dict'>
Var Name: alcohol misuse
Attributes: {'cld': ['FM', 'PA', 'C76', 'JUSTICE', 'DSS', 'E', 'NAT', 'FC', 'Full', 'Reentry'], 'degree': [9, 10, 4, 11, 7, 6, 12, 15, 12, 3], 'personal': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'institutional': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'relational': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'cultural': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'community': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'color': ['blue'], 'click': ['\nNode Features:\nCLD: FM\nLevel: org'], 'title': ['\nNode Features:\nCLD: FM\nLevel: org'], 'single_level': ['org', 'org', 'org', 'org', 'org', 'org', 'org', 'org', 'org', 'org'], 'level': 'org', 'nums_cld': 10}
Degree: 44
Type: <class 'numpy.str_'> <class 'dict'>
Var Name: mental health
Attributes: {'cld': ['FM', 'PA', 'FullGMB', 'C76', 'Wavi', 'JUSTICE', 'DSS', 'E', '

In [ ]:
# pickle.dump(combined_graph, open('Graph_union.pickle', 'wb'))

cld_nx.gravis_settings(clean_combined_graph)


gv.d3(
    clean_combined_graph, 
    graph_height=600,
    # edge_curvature=-0.5,
    
    # node_size_data_source="degree",
    # use_node_size_normalization=True,
    # node_size_normalization_min=15,
    # node_size_normalization_max=35,
    # show_node_label=True,
    # many_body_force_strength=-500
 ) 

# fig.export_html('Graph_union.html')


In [ ]:
# get trimmed graph 2 overlap nodes 
Graph_1_0 = cld_nx.clean_combined_graph(combined_graph, num_aggregate_nodes = 0, num_aggregate_edges= 0)
cld_nx.cld_nx_properties(Graph_1_0)


Graph data:
-------------
Type: <class 'networkx.classes.digraph.DiGraph'>
Directed: True
Number of nodes: 754
Number of edges: 1474


In [ ]:
# cld_nx.gravis_settings(Graph_1_0)

gv.d3(
    Graph_1_0, 
    graph_height=500,
    edge_curvature=-0.5,
    
    node_size_data_source="degree",
    use_node_size_normalization=True,
    node_size_normalization_min=15,
    node_size_normalization_max=35,
    show_node_label=True,
    many_body_force_strength=-500
 ) 


In [ ]:
fig = gv.three(Graph_1_0)
fig.export_html('Graph_1_0.html')

In [ ]:
# save graph object to file
# pickle.dump(Graph_1_0, open('Graph_1_0.pickle', 'wb'))

# load graph object from file
# G = pickle.load(open('filename.pickle', 'rb'))

In [ ]:
# get trimmed graph 2 overlap nodes 
Graph_0_1 = cld_nx.clean_combined_graph(combined_graph, num_aggregate_nodes = 0, num_aggregate_edges= 1)
cld_nx.cld_nx_properties(Graph_0_1)

In [ ]:
cld_nx.cld_nx_node_properties(Graph_0_1)

In [ ]:
cld_nx.gravis_settings(Graph_0_1)

fit = gv.d3(
    Graph_0_1, 
    graph_height=500,
    edge_curvature=-0.5,
    
    node_size_data_source="degree",
    use_node_size_normalization=True,
    node_size_normalization_min=15,
    node_size_normalization_max=35,
    show_node_label=True,
    many_body_force_strength=-500
 ) 

fig.export_html('Graph_0_1.html')

# CLD analysis 

Conduct analysis on the merged CLD. 

TODO: 
1. Find loops 
2. Find difference between 
2. Compare the merged CLD with the handcraft CLD
    - Read Arielle's handcraft CLD 
    - compare the shape between the two 


In [ ]:
graph_cycle  = sorted(nx.simple_cycles(Graph_1_0))


In [ ]:
Graph_1_0

In [ ]:
graph_cycle


In [ ]:
subgraph = Graph_1_0.subgraph(graph_cycle[2]).copy()

edge_weight = cld_nx.mulitiple_weights(subgraph)
for nodes, attrs in subgraph.nodes(data=True):
    print(nodes, attrs)

for u, v, attrs in subgraph.edges(data=True):
    print(u, v, attrs)


In [ ]:
cld_nx.gravis_settings(subgraph)

fig = gv.d3(
    subgraph, 
    graph_height=500,
    edge_curvature=-0.5,
    
    node_size_data_source="degree",
    use_node_size_normalization=True,
    node_size_normalization_min=15,
    node_size_normalization_max=35,
    show_node_label=True,
    many_body_force_strength=-500
 ) 

# fig.export_html('graph_cycle[2].html')

In [ ]:
# subgraph = Graph_1_0.subgraph(graph_cycle[9]).copy()

for i in range(len(graph_cycle)):
    subgraph = Graph_1_0.subgraph(graph_cycle[i]).copy()

    print(subgraph)

    edge_weight = cld_nx.mulitiple_weights(subgraph)
    print(edge_weight)


In [ ]:
graph_cycle[0]

# grab the weights of each cycle
# for u, v in graph_cycle[0]:
#     print(u, v, Graph_1_0[u][v]['weight'])


for u, v , attrs in Graph_1_0.edges(data = True):
    print(u, v, attrs['weight'])
    
    
        
    # print(u, v, Graph_1_0[u][v]['weight'])

# weights = [Graph_1_0[u][v]['weight'] for u, v in graph_cycle[0]]
# weights

In [ ]:
# find cycle 
cycles = nx.find_cycle(Graph_0_1)
cycles

sorted(nx.simple_cycles(Graph_0_1))


In [ ]:
cld_nx.cld_nx_properties(Graph_1_0)
cld_nx.cld_nx_node_properties(Graph_1_0)
cld_nx.cld_nx_edge_properties(Graph_1_0)

In [ ]:
cld_nx.cld_nx_edge_properties(Graph_1_0)

In [ ]:
for name, nx_graph in dict_cld.items(): 
    print(name)
    cycles = nx.find_cycle(nx_graph)
    print(cycles)